# Analisis Sentimen Teks Bahasa Indonesia

Repositori ini berisi implementasi pipeline Machine Learning untuk klasifikasi sentimen (positif dan negatif) pada teks ulasan berbahasa Indonesia.

## 📌 Alur Pengerjaan:
1. **Persiapan Lingkungan & Pustaka**: Import pandas, numpy, scikit-learn, joblib, matplotlib, seaborn.
2. **Pemuatan Data**: Membaca dataset ulasan dari CSV/sumber eksternal.
3. **Exploratory Data Analysis (EDA)**: Distribusi kelas dan karakteristik teks.
4. **Pra-pemrosesan Teks & Rekayasa Fitur**: Ekstraksi fitur menggunakan TF-IDF (Term Frequency - Inverse Document Frequency) dengan n-gram.
5. **Pelatihan Model**: Melatih model klasifikasi (Logistic Regression / Naive Bayes).
6. **Evaluasi Model**: Akurasi, Precision, Recall, F1-Score, dan Confusion Matrix.
7. **Penyimpanan Model**: Menyimpan model terlatih ke `model/model_sentimen.joblib`.
8. **Pengujian Inferensi**: Menguji model dengan teks ulasan baru secara real-time.


In [ ]:
# 1. Pemuatan Pustaka yang Dibutuhkan
import os
import gzip
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print('Semua pustaka berhasil diimpor!')


In [ ]:
# 2. Pemuatan Dataset
# Kita menggunakan dataset ulasan bahasa Indonesia
data_path = 'data/ulasan.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f'Dataset berhasil dimuat dari {data_path}')
else:
    print('Mengunduh / membuat dataset fallback...')
    # Data contoh ulasan sentimen bahasa Indonesia
    data_sample = {
        'text': [
            'pelayanan sangat memuaskan dan produk cepat sampai',
            'sangat kecewa barang rusak dan penjual tidak ramah',
            'kualitas luar biasa bagus, sangat direkomendasikan!',
            'aplikasi sering crash dan loading sangat lama sekali',
            'menurut saya, filmnya sama sekali tidak payah',
            'alur ceritanya membosankan dan tidak menarik',
            'fitur-fiturnya sangat membantu pekerjaan harian saya',
            'pengiriman sangat lambat, paket rusak saat tiba'
        ],
        'label': [1, 0, 1, 0, 1, 0, 1, 0]
    }
    df = pd.DataFrame(data_sample)

df.head()


In [ ]:
# 3. Exploratory Data Analysis (EDA)
print(f'Total baris data: {len(df)}')
print('\nDistribusi label sentimen:')
label_counts = df['label'].value_counts()
print(label_counts)

# Visualisasi distribusi label
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='Set2')
plt.title('Distribusi Sentimen (0 = Negatif, 1 = Positif)')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah')
plt.show()


In [ ]:
# 4. Pembagian Data (Train & Test Split)
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Jumlah data latih (train): {len(X_train)}')
print(f'Jumlah data uji (test)  : {len(X_test)}')


In [ ]:
# 5. Pembangunan Pipeline & Pelatihan Model
# Menggabungkan TF-IDF n-gram (1,2) dengan Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

# Melatih pipeline
pipeline.fit(X_train, y_train)
print('Pelatihan model berhasil selesai!')


In [ ]:
# 6. Evaluasi Model
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'=== Akurasi Model: {acc * 100:.2f}% ===\n')
print('Laporan Klasifikasi:')
print(classification_report(y_test, y_pred, target_names=['Negatif', 'Positif']))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Negatif', 'Pred Positif'],
            yticklabels=['Aktual Negatif', 'Aktual Positif'])
plt.title('Confusion Matrix')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.show()


In [ ]:
# 7. Menyimpan Model ke Direktori model/
os.makedirs('model', exist_ok=True)
model_file = 'model/model_sentimen.joblib'
joblib.dump(pipeline, model_file)
print(f'Model berhasil disimpan di: {model_file}')


In [ ]:
# 8. Pengujian Inferensi Real-Time
# Memuat kembali model tersimpan
loaded_model = joblib.load(model_file)

kalimat_uji = [
    'aplikasi ini sangat bermanfaat dan responsif sekali',
    'pelayanan jelek, pesanan datang terlambat dan tidak sesuai',
    'fiturnya cukup bagus tapi sering keluar sendiri',
    'menurut saya film ini sama sekali tidak payah, sangat keren!'
]

hasil_prediksi = loaded_model.predict(kalimat_uji)
probabilitas = loaded_model.predict_proba(kalimat_uji)

print('=== HASIL PREDIKSI SENTIMEN ===')
for teks, pred, prob in zip(kalimat_uji, hasil_prediksi, probabilitas):
    label_str = 'POSITIF (1)' if pred == 1 else 'NEGATIF (0)'
    confidence = max(prob) * 100
    print(f'Teks       : "{teks}"')
    print(f'Sentimen   : {label_str} (Tingkat Keyakinan: {confidence:.1f}%)')
    print('-' * 60)
